# Análisis predictivo de fallas en bomba centrífuga mediante datos de sensores

**Autor:** Diego [completar apellido]  
**Curso:** Programación y análisis reproducible con datos  
**Lenguaje principal:** Python  
**Dataset:** `pump_sensor_data_large.csv`  
**Producto complementario:** Dashboard interactivo en Streamlit  

Este notebook desarrolla un análisis reproducible de datos de sensores asociados a una bomba centrífuga industrial. El objetivo es identificar patrones operativos relacionados con eventos de falla y evaluar modelos básicos de clasificación supervisada para apoyar decisiones de mantenimiento predictivo.


## 1. Introducción y pregunta de análisis

Las bombas centrífugas son equipos críticos en sistemas industriales de transporte de fluidos. Su operación puede verse afectada por condiciones anómalas asociadas con vibración, temperatura, presión, caudal o consumo de potencia. El monitoreo de estas señales permite construir aproximaciones iniciales de mantenimiento predictivo.

**Pregunta principal de análisis**

> ¿Qué patrones operativos permiten identificar eventos de falla en una bomba centrífuga y qué tan efectivo es un modelo de clasificación supervisada para detectar dichas condiciones a partir de datos de sensores?

**Preguntas secundarias**

1. ¿Qué tan frecuente es la falla frente a la operación normal?
2. ¿Qué sensores presentan diferencias claras entre condición normal y falla?
3. ¿Existen patrones temporales asociados a los eventos de falla?
4. ¿Qué variables tienen mayor relación con la falla?
5. ¿Puede un modelo básico detectar fallas de forma confiable?


## 2. Objetivos y alcance

### Objetivo general

Desarrollar un análisis reproducible e interactivo de datos de sensores de una bomba centrífuga industrial, orientado a identificar patrones operativos asociados a fallas y evaluar modelos básicos de clasificación para apoyar decisiones de mantenimiento predictivo.

### Objetivos específicos

1. Cargar y documentar el dataset de sensores.
2. Explorar la estructura, calidad y distribución de las variables.
3. Analizar el desbalance entre operación normal y eventos de falla.
4. Visualizar el comportamiento temporal de las señales de sensores.
5. Comparar las variables operativas entre condiciones normales y de falla.
6. Crear variables derivadas mediante feature engineering temporal y operacional.
7. Entrenar modelos básicos de clasificación supervisada.
8. Evaluar el desempeño del modelo con métricas adecuadas para clases desbalanceadas.
9. Complementar el análisis con una aplicación interactiva en Streamlit.
10. Documentar el proceso, las decisiones analíticas, el uso de IA y las limitaciones del estudio.

### Alcance

El análisis permite identificar patrones asociados con registros etiquetados como falla. No pretende demostrar causalidad física definitiva ni validar industrialmente un sistema de mantenimiento predictivo real, debido a que no se dispone de información completa sobre fabricante, condiciones de operación, historial de mantenimiento ni curva oficial del equipo.


## 3. Carga de librerías y configuración

En esta sección se cargan las librerías necesarias para análisis de datos, visualización, pruebas estadísticas y modelado. Se fija una semilla aleatoria para asegurar reproducibilidad.


In [ ]:
# Manejo de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Estadística
from scipy import stats

# Modelado
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Configuración general
RANDOM_STATE = 42
TEST_SIZE = 0.20

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)


## 4. Carga del dataset

En esta sección se importa el archivo `pump_sensor_data_large.csv` y se realiza una primera inspección de su estructura.  
El objetivo es conservar una copia cruda del dataset en `df_raw`, sin aplicar todavía limpieza ni transformaciones. Esto permite mantener trazabilidad entre los datos originales y las versiones procesadas que se construirán en las siguientes secciones.

Para que el notebook sea reproducible en diferentes entornos, se prueban rutas relativas típicas de ejecución:

- `data/pump_sensor_data_large.csv`, si el notebook se ejecuta desde la raíz del repositorio.
- `../data/pump_sensor_data_large.csv`, si el notebook se ejecuta desde la carpeta `notebooks/`.


In [ ]:
from pathlib import Path

# Rutas candidatas según el punto desde donde se ejecute el notebook
candidate_paths = [
    Path("data/pump_sensor_data_large.csv"),
    Path("../data/pump_sensor_data_large.csv"),
    Path("/mnt/data/pump_sensor_data_large.csv")  # Ruta útil en este entorno de trabajo
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "No se encontró el archivo pump_sensor_data_large.csv. "
        "Verifica que esté ubicado en la carpeta data/ del repositorio."
    )

df_raw = pd.read_csv(DATA_PATH)

print(f"Dataset cargado desde: {DATA_PATH}")
print(f"Filas: {df_raw.shape[0]:,}")
print(f"Columnas: {df_raw.shape[1]}")

df_raw.head()


In [ ]:
# Estructura general del dataset
print("Información general del dataset:")
df_raw.info()


In [ ]:
# Resumen de columnas, tipos de dato y valores únicos
estructura_df = pd.DataFrame({
    "columna": df_raw.columns,
    "tipo_dato": df_raw.dtypes.astype(str).values,
    "valores_no_nulos": df_raw.notna().sum().values,
    "valores_unicos": df_raw.nunique().values
})

estructura_df


In [ ]:
# Estadísticas descriptivas iniciales de las variables numéricas
df_raw.describe().T.round(3)


In [ ]:
# Vista preliminar de la variable objetivo
target_preview = (
    df_raw["failure"]
    .value_counts()
    .rename_axis("failure")
    .reset_index(name="conteo")
)

target_preview["porcentaje"] = (target_preview["conteo"] / len(df_raw) * 100).round(2)
target_preview


### Interpretación inicial de la carga

El dataset fue cargado correctamente y contiene **100.000 registros** y **7 columnas**. La estructura inicial muestra una variable temporal (`timestamp`), cinco variables numéricas de sensores (`vibration`, `temperature`, `flow_rate`, `pressure` y `power_consumption`) y una variable objetivo binaria (`failure`).

La columna `timestamp` aparece inicialmente como tipo texto, por lo que más adelante debe convertirse a formato fecha/hora para habilitar el análisis temporal. Las variables de sensores aparecen como numéricas continuas, lo cual permite aplicar análisis exploratorio, visualizaciones, pruebas estadísticas, escalamiento y modelos de clasificación.

La variable `failure` identifica la condición operativa del registro: `0` para operación normal y `1` para falla. La revisión preliminar muestra que las fallas representan una proporción minoritaria del dataset, por lo que el problema debe tratarse como un caso de **clasificación binaria con clases desbalanceadas**. Por esta razón, en la etapa de modelado no será suficiente usar únicamente `accuracy`; también deberán evaluarse métricas como `precision`, `recall`, `F1-score`, matriz de confusión y ROC-AUC.


## 5. Inspección de calidad de datos

Se revisan valores faltantes, duplicados, tipos de datos, valores únicos y distribución de la variable objetivo `failure`. Aunque el dataset parezca limpio, esta revisión debe quedar documentada para cumplir con el criterio de reproducibilidad y trazabilidad del análisis.


In [ ]:
# Valores faltantes por columna
missing_summary = df_raw.isna().sum().to_frame("n_missing")
missing_summary["pct_missing"] = 100 * missing_summary["n_missing"] / len(df_raw)
missing_summary


In [ ]:
# Duplicados
duplicados = df_raw.duplicated().sum()
print(f"Registros duplicados: {duplicados:,}")


In [ ]:
# Valores únicos por columna
unique_summary = df_raw.nunique().to_frame("n_unique")
unique_summary


In [ ]:
# Distribución de la variable objetivo
target_counts = df_raw["failure"].value_counts().sort_index()
target_pct = df_raw["failure"].value_counts(normalize=True).sort_index() * 100

balance_df = pd.DataFrame({
    "n_registros": target_counts,
    "porcentaje": target_pct
})

balance_df


**Interpretación esperada:**  
El dataset está desbalanceado si la clase de falla representa una proporción baja frente a la operación normal. Por esta razón, la evaluación del modelo no debe depender únicamente de `accuracy`; también deben revisarse `precision`, `recall`, `F1-score`, matriz de confusión, ROC-AUC y PR-AUC.


## 6. Limpieza y transformación inicial

Se convierte la columna `timestamp` a formato fecha, se ordenan los registros cronológicamente y se valida que la variable objetivo sea binaria.


In [ ]:
df = df_raw.copy()

# Conversión temporal
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# Orden cronológico
df = df.sort_values("timestamp").reset_index(drop=True)

# Validaciones básicas
print("Tipo de timestamp:", df["timestamp"].dtype)
print("Valores únicos en failure:", sorted(df["failure"].unique()))
print("Fechas:", df["timestamp"].min(), "a", df["timestamp"].max())

df.head()


In [ ]:
# Revisión de rangos físicos o plausibles
sensor_cols = ["vibration", "temperature", "flow_rate", "pressure", "power_consumption"]

range_summary = df[sensor_cols].agg(["min", "max", "mean", "std"]).T
range_summary


**Interpretación pendiente:**  
Revisar si los rangos son plausibles para un caso académico de sensores. Señalar cualquier valor extremo o comportamiento llamativo. Si no se observan inconsistencias evidentes, dejarlo documentado explícitamente.


## 7. Feature engineering

Se crean variables derivadas para representar comportamiento temporal, cambios entre mediciones, relaciones operativas y tendencias móviles. Estas variables buscan aportar información adicional para detectar condiciones anómalas.


In [ ]:
# Variables temporales
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["elapsed_minutes"] = (df["timestamp"] - df["timestamp"].min()).dt.total_seconds() / 60

# Representación cíclica de la hora
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Diferencias entre registros consecutivos
for col in sensor_cols:
    df[f"{col}_diff"] = df[col].diff().fillna(0)

# Ratios operativos. Se agrega epsilon para evitar división por cero.
eps = 1e-6
df["power_flow_ratio"] = df["power_consumption"] / (df["flow_rate"] + eps)
df["pressure_flow_ratio"] = df["pressure"] / (df["flow_rate"] + eps)
df["vibration_power_ratio"] = df["vibration"] / (df["power_consumption"] + eps)

# Variables móviles con ventana de 10 minutos
window = 10
for col in ["vibration", "temperature", "pressure", "power_consumption"]:
    df[f"{col}_roll_mean_{window}"] = df[col].rolling(window=window, min_periods=1).mean()
    df[f"{col}_roll_std_{window}"] = df[col].rolling(window=window, min_periods=1).std().fillna(0)

print(f"Columnas después de feature engineering: {df.shape[1]}")
df.head()


In [ ]:
# Lista final de variables derivadas
engineered_cols = [col for col in df.columns if col not in df_raw.columns]
engineered_cols


**Interpretación esperada:**  
Las variables instantáneas describen el estado de la bomba en un momento específico. Las variables derivadas incorporan tendencia, variabilidad y relaciones operativas que pueden mejorar la detección de fallas frente al uso exclusivo de señales instantáneas.


## 8. Análisis exploratorio de datos

Esta sección debe incluir al menos cinco visualizaciones relevantes, cada una con título, etiquetas e interpretación. El objetivo es conectar los gráficos con la pregunta de análisis, no solo mostrar figuras.


### 8.1 Balance de clases

Este gráfico muestra la proporción entre registros normales y registros de falla.


In [ ]:
fig = px.bar(
    balance_df.reset_index().rename(columns={"index": "failure"}),
    x="failure",
    y="n_registros",
    text="porcentaje",
    title="Distribución de clases: operación normal vs falla",
    labels={"failure": "Clase failure", "n_registros": "Número de registros"}
)
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.show()


**Interpretación pendiente:**  
Describir la proporción de fallas y explicar por qué esto afecta la selección de métricas de evaluación.


### 8.2 Serie temporal interactiva de sensores

Este gráfico permite observar la evolución temporal de las señales principales y ubicar visualmente eventos de falla.


In [ ]:
sensor_to_plot = "vibration"  # Cambiar por: temperature, flow_rate, pressure, power_consumption

fig = px.line(
    df,
    x="timestamp",
    y=sensor_to_plot,
    color=df["failure"].astype(str),
    title=f"Serie temporal de {sensor_to_plot} coloreada por condición de falla",
    labels={"timestamp": "Tiempo", sensor_to_plot: sensor_to_plot, "color": "failure"}
)
fig.show()


**Interpretación pendiente:**  
Indicar si los eventos de falla se concentran en ciertos niveles del sensor seleccionado o si aparecen distribuidos aleatoriamente en el tiempo.


### 8.3 Comparación de sensores por clase

Los boxplots permiten comparar el comportamiento de cada sensor en operación normal y en falla.


In [ ]:
df_long = df.melt(
    id_vars=["failure"],
    value_vars=sensor_cols,
    var_name="sensor",
    value_name="value"
)

fig = px.box(
    df_long,
    x="sensor",
    y="value",
    color=df_long["failure"].astype(str),
    title="Comparación de sensores por condición normal/falla",
    labels={"sensor": "Sensor", "value": "Valor", "color": "failure"}
)
fig.show()


**Interpretación pendiente:**  
Identificar qué sensores muestran mayor separación entre registros normales y fallas.


### 8.4 Distribución de sensores clave

Se comparan histogramas por clase para sensores seleccionados.


In [ ]:
sensor_to_hist = "vibration"  # Cambiar según interés

fig = px.histogram(
    df,
    x=sensor_to_hist,
    color=df["failure"].astype(str),
    nbins=60,
    marginal="box",
    opacity=0.70,
    title=f"Distribución de {sensor_to_hist} por clase",
    labels={sensor_to_hist: sensor_to_hist, "color": "failure"}
)
fig.show()


**Interpretación pendiente:**  
Describir si existe solapamiento entre clases o si la variable separa claramente condiciones normales y fallas.


### 8.5 Matriz de correlación

La matriz de correlación permite observar relaciones lineales entre sensores, variables derivadas y la variable objetivo.


In [ ]:
corr_cols = sensor_cols + [
    "power_flow_ratio",
    "pressure_flow_ratio",
    "vibration_power_ratio",
    "vibration_roll_mean_10",
    "temperature_roll_mean_10",
    "failure"
]

corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlación entre sensores, variables derivadas y falla")
plt.tight_layout()
plt.show()


**Interpretación pendiente:**  
Señalar qué variables tienen mayor asociación lineal con `failure`. Recordar que correlación no implica causalidad.


### 8.6 PCA 2D para visualización multivariable

Se utiliza PCA para proyectar las variables numéricas en dos componentes y observar si existe separación visual entre clases.


In [ ]:
feature_cols_for_pca = sensor_cols + engineered_cols

X_pca = df[feature_cols_for_pca].replace([np.inf, -np.inf], np.nan).fillna(0)

scaler_pca = StandardScaler()
X_scaled_pca = scaler_pca.fit_transform(X_pca)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_components = pca.fit_transform(X_scaled_pca)

pca_df = pd.DataFrame({
    "PC1": pca_components[:, 0],
    "PC2": pca_components[:, 1],
    "failure": df["failure"].astype(str)
})

fig = px.scatter(
    pca_df.sample(n=min(10000, len(pca_df)), random_state=RANDOM_STATE),
    x="PC1",
    y="PC2",
    color="failure",
    title="Proyección PCA 2D de sensores y variables derivadas",
    labels={"failure": "failure"}
)
fig.show()

print("Varianza explicada por PC1 y PC2:", pca.explained_variance_ratio_)


**Interpretación pendiente:**  
Comentar si las clases aparecen mezcladas o separadas en el espacio reducido. Si hay separación fuerte, discutir si el dataset puede ser sintético o altamente separable.


## 9. Análisis estadístico

Se aplica una prueba estadística para evaluar si existen diferencias significativas entre registros normales y registros de falla en sensores clave. Esta prueba no demuestra causalidad, pero respalda cuantitativamente las diferencias observadas en el EDA.


In [ ]:
def mann_whitney_test_by_failure(data, variable, target="failure"):
    normal = data.loc[data[target] == 0, variable]
    failure = data.loc[data[target] == 1, variable]

    stat, p_value = stats.mannwhitneyu(normal, failure, alternative="two-sided")

    return {
        "variable": variable,
        "normal_median": normal.median(),
        "failure_median": failure.median(),
        "statistic": stat,
        "p_value": p_value
    }

test_results = pd.DataFrame([
    mann_whitney_test_by_failure(df, col) for col in sensor_cols
])

test_results


**Interpretación pendiente:**  
Si el p-valor es menor que 0.05, se rechaza la hipótesis nula de igualdad de distribuciones entre operación normal y falla para esa variable. Debe interpretarse junto con la magnitud de la diferencia y no solo con significancia estadística.


## 10. Preparación para modelado

Se define la matriz de variables predictoras `X` y la variable objetivo `y`. La división train/test se realiza de forma estratificada para preservar la proporción de fallas en ambos conjuntos.


In [ ]:
# Selección de variables predictoras
feature_cols = sensor_cols + engineered_cols

# Limpieza final de infinitos o NaN generados por ratios
model_df = df[feature_cols + ["failure"]].replace([np.inf, -np.inf], np.nan).fillna(0)

X = model_df[feature_cols]
y = model_df["failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Distribución y_train:")
print(y_train.value_counts(normalize=True))
print("Distribución y_test:")
print(y_test.value_counts(normalize=True))


## 11. Modelo baseline

El modelo baseline predice siempre la clase mayoritaria. Sirve para mostrar que, en un problema desbalanceado, una alta exactitud puede ser engañosa si el modelo no detecta fallas.


In [ ]:
def evaluate_classifier(name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_score)
        pr_auc = average_precision_score(y_test, y_score)
    else:
        roc_auc = np.nan
        pr_auc = np.nan

    metrics = {
        "modelo": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_falla": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "recall_falla": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "f1_falla": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc
    }

    return metrics, y_pred

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_metrics, baseline_pred = evaluate_classifier("Baseline clase mayoritaria", baseline, X_test, y_test)
baseline_metrics


In [ ]:
print(classification_report(y_test, baseline_pred, zero_division=0))


**Interpretación pendiente:**  
Explicar por qué el modelo baseline puede tener accuracy alta, pero recall de falla igual a cero o muy bajo.


## 12. Regresión logística

La regresión logística se utiliza como modelo interpretable de clasificación binaria. Permite estimar la probabilidad de falla a partir de sensores y variables derivadas.


In [ ]:
logistic_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000,
        class_weight="balanced"
    ))
])

logistic_model.fit(X_train, y_train)

logistic_metrics, logistic_pred = evaluate_classifier("Regresión logística", logistic_model, X_test, y_test)
logistic_metrics


In [ ]:
print(classification_report(y_test, logistic_pred, zero_division=0))

cm = confusion_matrix(y_test, logistic_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Falla"])
disp.plot(cmap="Blues")
plt.title("Matriz de confusión - Regresión logística")
plt.show()


In [ ]:
# Coeficientes de regresión logística
coef = logistic_model.named_steps["model"].coef_[0]
coef_df = pd.DataFrame({
    "variable": feature_cols,
    "coeficiente": coef,
    "abs_coeficiente": np.abs(coef)
}).sort_values("abs_coeficiente", ascending=False)

coef_df.head(15)


**Interpretación pendiente:**  
Analizar las variables con mayor magnitud de coeficiente. Recordar que el signo indica dirección de asociación con la probabilidad de falla en el espacio escalado, no causalidad física definitiva.


## 13. Modelo comparativo: Random Forest

Se entrena un Random Forest como modelo no lineal de comparación. También permite obtener una medida de importancia de variables.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_metrics, rf_pred = evaluate_classifier("Random Forest", rf_model, X_test, y_test)
rf_metrics


In [ ]:
print(classification_report(y_test, rf_pred, zero_division=0))

cm = confusion_matrix(y_test, rf_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Falla"])
disp.plot(cmap="Greens")
plt.title("Matriz de confusión - Random Forest")
plt.show()


In [ ]:
importance_df = pd.DataFrame({
    "variable": feature_cols,
    "importancia": rf_model.feature_importances_
}).sort_values("importancia", ascending=False)

importance_df.head(15)


**Interpretación pendiente:**  
Comparar si las variables más importantes coinciden con los hallazgos del EDA y con los coeficientes de la regresión logística.


## 14. Comparación de modelos

Se comparan los modelos evaluados usando métricas adecuadas para clasificación desbalanceada.


In [ ]:
metrics_df = pd.DataFrame([
    baseline_metrics,
    logistic_metrics,
    rf_metrics
])

metrics_df


In [ ]:
fig = px.bar(
    metrics_df.melt(id_vars="modelo", var_name="metrica", value_name="valor"),
    x="metrica",
    y="valor",
    color="modelo",
    barmode="group",
    title="Comparación de métricas de clasificación por modelo",
    labels={"metrica": "Métrica", "valor": "Valor", "modelo": "Modelo"}
)
fig.show()


**Interpretación pendiente:**  
Indicar cuál modelo es más conveniente para el problema y justificarlo. En mantenimiento predictivo, el recall de la clase falla suele ser crítico porque representa la capacidad de detectar fallas reales.


## 15. Dashboard en Streamlit

Como complemento al notebook, se desarrollará una aplicación en Streamlit que permita explorar los datos de forma interactiva.

**Módulos esperados de la app:**

1. Resumen del dataset.
2. Exploración temporal de sensores.
3. Comparación entre operación normal y falla.
4. Variables derivadas de feature engineering.
5. Modelado y métricas.
6. Conclusiones principales.

**Pendiente:** incluir enlace de la app desplegada una vez esté publicada.


## 16. Uso documentado de IA

La siguiente tabla documenta cómo se usaron herramientas de IA durante el proyecto. La idea no es ocultar el uso de IA, sino demostrar criterio propio al revisar, corregir y adaptar sus respuestas.

| Herramienta | Qué se pidió | Qué entregó | Qué se ajustó | Validación propia |
|---|---|---|---|---|
| ChatGPT | Estructura del proyecto final | Plan de trabajo y secciones | Se adaptó al dataset de sensores | Se contrastó con la rúbrica del curso |
| ChatGPT | Ideas de feature engineering | Variables temporales, ratios y medias móviles | Se seleccionaron variables interpretables | Se verificó que fueran calculables con el dataset |
| ChatGPT/Copilot | Apoyo para código de gráficos/modelado | Fragmentos de código base | Se corrigieron nombres de columnas y rutas | Se ejecutó el notebook de principio a fin |
| ChatGPT | Interpretación de métricas | Texto explicativo | Se ajustó para evitar sobreinterpretación | Se contrastó con matriz de confusión y métricas reales |


## 17. Conclusiones

### Respuesta a la pregunta principal

**Pendiente de completar después del análisis final.**

Se espera responder de forma directa:

- Qué patrones operativos permiten identificar fallas.
- Qué sensores resultan más relevantes.
- Qué tan bien funcionan los modelos evaluados.
- Qué limitaciones tiene el análisis.

### Limitaciones

1. El dataset se interpreta como un caso académico o demostrativo si no se dispone de procedencia industrial completa.
2. El análisis detecta asociaciones entre sensores y falla, pero no prueba causalidad física.
3. No se modela tiempo restante hasta falla.
4. No se validan umbrales contra normas, fabricante o historial real de mantenimiento.
5. Los modelos requieren validación con datos reales adicionales antes de uso operacional.

### Recomendación concreta

**Pendiente:** formular una recomendación basada en los hallazgos finales. Por ejemplo, priorizar monitoreo de sensores con mayor asociación a falla, complementar con ventanas móviles y evaluar el modelo con énfasis en recall de falla.


## 18. Checklist final antes de entregar

- [ ] El notebook corre de principio a fin sin errores.
- [ ] La pregunta de análisis está claramente formulada.
- [ ] La limpieza está documentada con justificación.
- [ ] Hay al menos cinco visualizaciones con título, etiquetas e interpretación.
- [ ] Hay al menos una prueba estadística o un modelo con interpretación.
- [ ] Las conclusiones responden la pregunta principal.
- [ ] El README explica el proyecto y cómo reproducirlo.
- [ ] Existe `requirements.txt`.
- [ ] El repositorio tiene al menos tres commits significativos.
- [ ] La app Streamlit funciona y tiene gráficos interactivos.
- [ ] El uso de IA está documentado críticamente.
